# Prompt Evaluation Workflow

1. **Draft a Prompt**
   - Write the initial prompt and define the expected output.

2. **Create an Evaluation Dataset**
   - Prepare test cases with sample inputs and expected outputs.

3. **Feed Through Claude**
   - Run the prompt through Claude using the evaluation dataset.
   - Collect and record the generated outputs.

4. **Feed Through Grader**
   - Evaluate Claude's outputs against predefined criteria.
   - Measure accuracy, relevance, consistency, and instruction adherence.

5. **Change Prompt and Repeat**
   - Analyze evaluation results and identify weaknesses.
   - Refine the prompt based on the grader's feedback.
   - Repeat the evaluation workflow until the desired performance is achieved.

In [21]:
# Load env variables from .env file
from dotenv import load_dotenv

load_dotenv()
import os


In [22]:
# Create API Client
from anthropic import Anthropic

client = Anthropic(
    base_url = os.getenv("ANTHROPIC_BASE_URL"),
    api_key = os.getenv("ANTHROPIC_API_KEY")
)

model_name = os.getenv("ANTHROPIC_MODEL")

In [23]:
# Helper Functions 
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model_name,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text


In [24]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [25]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [26]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [27]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [28]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [29]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [30]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [31]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 3.5


In [32]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n\\bus-east-1\\b|ap-northeast-1|us-east-2|eu-west-1|eu-central-1|ap-northeast-2|ap-south-1|ca-central-1|ap-southeast-1|ap-southeast-2|eu-west-2|sa-east-1|us-west-1|us-west-2\\b\n",
    "test_case": {
      "task": "Retrieve a list of all AWS regions supported by AWS SDK",
      "format": "python",
      "solution_criteria": "A single Python function that returns a list of AWS regions using boto3 or another AWS SDK."
    },
    "score": 1.0,
    "reasoning": "The solution lacks a Python function to retrieve AWS regions and does not use an AWS SDK, which are essential requirements."
  },
  {
    "output": "_json\n[\n    {\n        \"InstanceId\": \"i-1234567890abcdef0\",\n        \"InstanceName\": \"WebServer1\"\n    },\n    {\n        \"InstanceId\": \"i-234567890abcdef01\",\n        \"InstanceName\": \"DBServer\"\n    }\n]\n",
    "test_case": {
      "task": "Convert a list of AWS EC2 instance names into a JSON object",
      "format": "json",
      "solution_cri